In [14]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:

data = pd.read_csv(r"C:\Users\hp\Downloads\ML\IMDB_dataset.csv")

In [3]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [5]:
data.shape

(50000, 2)

In [6]:
data['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [15]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

data['review'] = data['review'].apply(
    lambda x: ' '.join(word.lower() for word in word_tokenize(x) if word.isalpha())
)

data['review'] = data['review'].apply(
    lambda x: ' '.join(word for word in x.split() if word not in stop_words)
)
lemmatizer = WordNetLemmatizer()
data['review'] = data['review'].apply(
    lambda x: ' '.join(lemmatizer.lemmatize(word) for word in x.split())
)

In [16]:
data['review']

0        one reviewer mentioned watching oz episode hoo...
1        wonderful little production br br filming tech...
2        thought wonderful way spend time hot summer we...
3        basically family little boy jake think zombie ...
4        petter mattei love time money visually stunnin...
                               ...                        
49995    thought movie right good job creative original...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    catholic taught parochial elementary school nu...
49998    going disagree previous comment side maltin on...
49999    one expects star trek movie high art fan expec...
Name: review, Length: 50000, dtype: object

In [17]:
x = data['review']
y = data['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    data['review'],
    data['sentiment'],
    test_size=0.2,
    random_state=42
)

bow = CountVectorizer()

x_train_bow = bow.fit_transform(x_train)
x_test_bow = bow.transform(x_test)

In [18]:
model = MultinomialNB()
model.fit(x_train_bow, y_train)
print("✅ Model trained successfully!")
print(f"   Classes: {list(model.classes_)}")
print(f"   Training samples: {x_train_bow.shape[0]}")

✅ Model trained successfully!
   Classes: [np.str_('negative'), np.str_('positive')]
   Training samples: 40000


In [19]:
y_pred = model.predict(x_test_bow)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8588
              precision    recall  f1-score   support

    negative       0.85      0.87      0.86      4961
    positive       0.87      0.84      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



In [24]:
print("=" * 55)
print("   SENTIMENT ANALYSIS RESULTS")
print("=" * 55)
print(f"\n🎯 Accuracy: {accuracy_score(y_test, y_pred):.4f} ({accuracy_score(y_test, y_pred)*100:.2f}%)")
print(f"\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

print("\n--- Confusion Matrix ---")
print(f"{'':15}{model.classes_[0]:>12}{model.classes_[1]:>12}")
print(f"{model.classes_[0]:15}{cm[0][0]:>12}{cm[0][1]:>12}")
print(f"{model.classes_[1]:15}{cm[1][0]:>12}{cm[1][1]:>12}")

   SENTIMENT ANALYSIS RESULTS

🎯 Accuracy: 0.8588 (85.88%)

--- Detailed Classification Report ---
              precision    recall  f1-score   support

    negative       0.85      0.87      0.86      4961
    positive       0.87      0.84      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000


--- Confusion Matrix ---
                   negative    positive
negative               4337         624
positive                788        4251


In [37]:
review = input("Enter a movie review: ")

print(f"review: {review}")

review = ' '.join(
    word.lower() for word in word_tokenize(review) if word.isalpha()
)

review = ' '.join(
    word for word in review.split() if word not in stop_words
)

review = ' '.join(lemmatizer.lemmatize(word) for word in review.split())


review_bow = bow.transform([review])


prediction = model.predict(review_bow)


print("\n" + "=" * 40)
print("        SENTIMENT PREDICTION")
print("=" * 40)

print(f"\n📝 Processed Review: {review}")

if prediction[0] == "positive":
    print("🟢 Sentiment: POSITIVE 😊")
else:
    print("🔴 Sentiment: NEGATIVE 😞")

print("=" * 40)

review: this movie is really amazing. i really enjoyed each and every part

        SENTIMENT PREDICTION

📝 Processed Review: movie really amazing really enjoyed every part
🟢 Sentiment: POSITIVE 😊
